# 🎯 LendingClub End-to-End Default Prediction Pipeline
This notebook implements an optimized machine learning solution utilizing Apache Spark. 
It ingests 2.2M+ rows, processes features in parallel, and compares multiple classifiers.

In [10]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, regexp_extract, when
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, MinMaxScaler
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier, MultilayerPerceptronClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
import pandas as pd

# Spark Engine Setup with Interactive Eager Notebook Evaluation enabled
spark = SparkSession.builder \
    .appName("lending-club") \
    .config("spark.sql.repl.eagerEval.enabled", True) \
    .config("spark.sql.repl.eagerEval.maxNumRows", 10) \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.ansi.enabled", "false")

## 📥 Data Ingestion & Null Cleansing
We load the targeted feature footprint and purge empty values across the distributed cluster.

In [11]:
# 💻 Code (Ingestion & Quality Cleanse)
selected_columns = [
    "id", "purpose", "term", "verification_status", "acc_now_delinq", "addr_state", 
    "annual_inc", "application_type", "dti", "grade", "home_ownership", "initial_list_status", 
    "installment", "int_rate", "loan_amnt", "loan_status", "tax_liens", "delinq_amnt", 
    "policy_code", "last_fico_range_high", "last_fico_range_low", "recoveries", "collection_recovery_fee"
]

path = '/kaggle/input/datasets/wordsforthewise/lending-club/accepted_2007_to_2018Q4.csv.gz'

# Load with native type inference and instantly drop missing blocks
df = spark.read.csv(path, header=True, inferSchema=True).select(selected_columns).na.drop()

## 🛠️ Feature Engineering & Target Mapping
Cleaning data ranges, pulling digits out of strings, mapping target arrays, and handling minority sampling structures.

In [12]:
# 💻 Code (Transformations & Stratified Downsampling)

# 1. Feature transformations & Target Mapping
df_prep = (
    df.withColumn("term", regexp_extract("term", r"\d+", 0).cast("int"))
      .withColumn("verification_status_encoded", when(col("verification_status").isin("Verified", "Source Verified"), 0).otherwise(1))
      .withColumn("acc_now_delinq", when(col("acc_now_delinq").cast("int") >= 4, 4).otherwise(col("acc_now_delinq").cast("int")))
      .withColumn("application_type", when(col("application_type") == "Joint App", 0).when(col("application_type") == "Individual", 1))
      .withColumn("target", when(col("loan_status") == "Fully Paid", 0)
                           .when(col("loan_status").isin("Late (16-30 days)", "Late (31-120 days)", "In Grace Period"), 1)
                           .when(col("loan_status").isin("Charged Off", "Default"), 2))
      .drop("verification_status", "loan_status", "id")
      .filter(col("acc_now_delinq").between(0, 4) & col("application_type").isNotNull() & col("target").isNotNull())
)

# 2. Downsample Class 0 to 30% & drop NaNs
df_sampled = df_prep.sampleBy("target", fractions={0: 0.3, 1: 1.0, 2: 1.0}, seed=42).na.drop()

## 🧬 High-Speed Matrix Vectorization
Text categories and numeric variables are processed concurrently in bulk vectors without slow single loops. One-Hot Encoding loops are removed to protect cluster memory.

In [13]:
# 💻 Code (Parallel Indexing, Casting, and Splitting)

# 1. Categorical Encoding (StringIndexer + OneHotEncoder)
cat_cols = [c for c in ['purpose', 'addr_state', 'home_ownership', 'initial_list_status', 'grade'] if c in df_sampled.columns]
idx_cols, vec_cols = [f"{c}_idx" for c in cat_cols], [f"{c}_vec" for c in cat_cols]

df_prep = StringIndexer(inputCols=cat_cols, outputCols=idx_cols).fit(df_sampled).transform(df_sampled)
df_prep = OneHotEncoder(inputCols=idx_cols, outputCols=vec_cols).fit(df_prep).transform(df_prep).drop(*cat_cols, *idx_cols)

# 2. Dynamic Type Casting & Vector Assembly
features = [c for c in df_prep.columns if c != "target"]
df_typed = df_prep.select([col(c).cast("double") if not c.endswith("_vec") else col(c) for c in df_prep.columns])
df_vector = VectorAssembler(inputCols=features, outputCol="raw_features").transform(df_typed)

# 3. 80/10/10 Split & Scaler Fit on Train Only
train, temp = df_vector.randomSplit([0.8, 0.2], seed=42)
val, test = temp.randomSplit([0.5, 0.5], seed=42)

scaler = MinMaxScaler(inputCol="raw_features", outputCol="features").fit(train)
train_scaled, val_scaled, test_scaled = scaler.transform(train), scaler.transform(val), scaler.transform(test)

## 🤖 Multi-Model Machine Learning Evaluation Matrix
We train and benchmark Logistic Regression, Random Forest, and a Multilayer Perceptron Deep Neural Network, reporting the metrics in an interactive dashboard table.

In [14]:
# 💻 Code (Model Training & Metric Comparison Layout)
# 1. Pre-load data splits directly into RAM to prevent evaluation pipeline lag
for ds in [train_scaled, val_scaled, test_scaled]: ds.cache().count()

# 2. Shared Evaluator instance (reused to avoid unnecessary object creation)
evaluator = MulticlassClassificationEvaluator(labelCol="target")

# 3. Model definitions (auto-scaled MLP input layers based on vector length)
models = [
    (LogisticRegression(featuresCol="features", labelCol="target"), "Logistic Regression"),
    (RandomForestClassifier(featuresCol="features", labelCol="target"), "Random Forest"),
    (MultilayerPerceptronClassifier(layers=[train_scaled.select("features").head()[0].size, 64, 32, 3], 
                                    featuresCol="features", labelCol="target", maxIter=100, seed=123), "Neural Network")
]

# 4. Sequential Fit & Single-Pass Evaluation
def evaluate(model, name):
    fit_m = model.fit(train_scaled)
    res = {"Model": name}
    for label, ds in [("Train", train_scaled), ("Validation", val_scaled), ("Test", test_scaled)]:
        preds = fit_m.transform(ds)  # Calculate predictions ONCE per split
        res[f"Accuracy ({label})"] = round(evaluator.setMetricName("accuracy").evaluate(preds), 3)
        res[f"F1 Score ({label})"] = round(evaluator.setMetricName("f1").evaluate(preds), 3)
    return res

# Render as a clean Pandas DataFrame table
pd.DataFrame([evaluate(m, name) for m, name in models])

,Model,Accuracy (Train),F1 Score (Train),Accuracy (Validation),F1 Score (Validation),Accuracy (Test),F1 Score (Test)
0,Logistic Regression,0.879,0.861,0.878,0.860,0.877,0.859
1,Random Forest,0.874,0.849,0.873,0.849,0.874,0.849
2,Neural Network,0.860,0.840,0.859,0.839,0.859,0.839
